# A/B-тестирование новой версии сайта

## Прикладной проект: Оценка эффективности изменения интерфейса

В этом ноутбуке мы применим знания из **Модуля 3 (Выводная статистика)** для анализа результатов A/B-теста интернет-магазина.

Компания изменила кнопку оформления заказа и хочет понять, действительно ли новая версия (**B**) увеличивает конверсию по сравнению со старой (**A**), или разница случайна.

### Содержание:
1. Загрузка и предобработка данных
2. Exploratory Data Analysis (EDA)
3. Доверительные интервалы для конверсий
4. Проверка гипотез: z-тест для двух пропорций
5. Размер эффекта и мощность теста
6. Байесовский взгляд на A/B-тест (Beta-Binomial)
7. Выводы и рекомендации

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Библиотеки успешно загружены!')

## 1. Загрузка и предобработка данных

Сгенерируем синтетические данные A/B-теста, имитирующие реальный трафик сайта.

### Структура данных:
- **user_id**: Уникальный идентификатор пользователя
- **group**: Группа теста (`A` — контроль, `B` — новая версия)
- **converted**: Совершил ли пользователь покупку (1) или нет (0)
- **time_on_site**: Время на сайте в секундах

In [ ]:
# Генерация синтетических данных A/B-теста
np.random.seed(42)

# Истинные (неизвестные аналитику) конверсии групп
true_conv_A = 0.120   # контроль
true_conv_B = 0.138   # новая версия (эффект +1.8 п.п.)

n_A = 5000  # пользователей в группе A
n_B = 5000  # пользователей в группе B

# Генерация конверсий (распределение Бернулли)
conv_A = np.random.binomial(1, true_conv_A, n_A)
conv_B = np.random.binomial(1, true_conv_B, n_B)

# Время на сайте (лог-нормальное распределение)
time_A = np.random.lognormal(mean=4.8, sigma=0.5, size=n_A)
time_B = np.random.lognormal(mean=4.9, sigma=0.5, size=n_B)

df = pd.DataFrame({
    'user_id': np.arange(1, n_A + n_B + 1),
    'group': ['A'] * n_A + ['B'] * n_B,
    'converted': np.concatenate([conv_A, conv_B]),
    'time_on_site': np.round(np.concatenate([time_A, time_B]), 1)
})

# Перемешаем строки
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print('Данные A/B-теста')
print('=' * 60)
print(f'Всего пользователей: {len(df)}')
print(f'Группа A: {(df["group"] == "A").sum()}')
print(f'Группа B: {(df["group"] == "B").sum()}')
print(f'\nПервые 10 строк:')
print(df.head(10))

## 2. Exploratory Data Analysis (EDA)

Оценим наблюдаемые конверсии и распределение времени на сайте.

In [ ]:
# Сводка по группам
summary = df.groupby('group').agg(
    users=('user_id', 'count'),
    conversions=('converted', 'sum'),
    conv_rate=('converted', 'mean'),
    avg_time=('time_on_site', 'mean')
)
summary['conv_rate_%'] = (summary['conv_rate'] * 100).round(2)

print('Сводная статистика по группам')
print('=' * 60)
print(summary)

p_A = summary.loc['A', 'conv_rate']
p_B = summary.loc['B', 'conv_rate']
print(f'\nНаблюдаемая конверсия A: {p_A*100:.2f}%')
print(f'Наблюдаемая конверсия B: {p_B*100:.2f}%')
print(f'Абсолютная разница: {(p_B - p_A)*100:.2f} п.п.')
print(f'Относительный прирост: {(p_B/p_A - 1)*100:.2f}%')

In [ ]:
# Визуализация EDA
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Конверсия по группам
conv_rates = [p_A * 100, p_B * 100]
axes[0, 0].bar(['A (контроль)', 'B (новая)'], conv_rates,
               color=['steelblue', 'coral'], alpha=0.7)
for i, v in enumerate(conv_rates):
    axes[0, 0].text(i, v + 0.1, f'{v:.2f}%', ha='center', fontweight='bold')
axes[0, 0].set_ylabel('Конверсия (%)')
axes[0, 0].set_title('Конверсия по группам')

# 2. Количество конверсий
convs = [summary.loc['A', 'conversions'], summary.loc['B', 'conversions']]
axes[0, 1].bar(['A', 'B'], convs, color=['steelblue', 'coral'], alpha=0.7)
axes[0, 1].set_ylabel('Количество покупок')
axes[0, 1].set_title('Абсолютное число конверсий')

# 3. Распределение времени на сайте
axes[1, 0].hist(df[df['group'] == 'A']['time_on_site'], bins=40, alpha=0.6,
                label='A', color='steelblue')
axes[1, 0].hist(df[df['group'] == 'B']['time_on_site'], bins=40, alpha=0.6,
                label='B', color='coral')
axes[1, 0].set_xlabel('Время на сайте (сек)')
axes[1, 0].set_ylabel('Количество')
axes[1, 0].set_title('Распределение времени на сайте')
axes[1, 0].legend()

# 4. Boxplot времени по группам
df.boxplot(column='time_on_site', by='group', ax=axes[1, 1])
axes[1, 1].set_xlabel('Группа')
axes[1, 1].set_ylabel('Время на сайте (сек)')
axes[1, 1].set_title('Время на сайте по группам')
plt.suptitle('')

plt.tight_layout()
plt.show()

## 3. Доверительные интервалы для конверсий

Конверсия — это доля (пропорция). Доверительный интервал для пропорции (нормальное приближение Вальда):

$$\hat{p} \pm z_{\alpha/2} \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

где $z_{\alpha/2}$ — квантиль стандартного нормального распределения (для 95% доверия $z \approx 1.96$).

In [ ]:
# Доверительные интервалы для пропорций (95%)
def proportion_ci(successes, n, confidence=0.95):
    p = successes / n
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    se = np.sqrt(p * (1 - p) / n)
    return p, p - z * se, p + z * se

s_A = int(summary.loc['A', 'conversions'])
s_B = int(summary.loc['B', 'conversions'])

p_A, lo_A, hi_A = proportion_ci(s_A, n_A)
p_B, lo_B, hi_B = proportion_ci(s_B, n_B)

print('95% доверительные интервалы для конверсии')
print('=' * 60)
print(f'Группа A: {p_A*100:.2f}%  ДИ = [{lo_A*100:.2f}%, {hi_A*100:.2f}%]')
print(f'Группа B: {p_B*100:.2f}%  ДИ = [{lo_B*100:.2f}%, {hi_B*100:.2f}%]')

# Визуализация ДИ
fig, ax = plt.subplots(figsize=(9, 5))
groups = ['A (контроль)', 'B (новая)']
means = [p_A * 100, p_B * 100]
errors = [[(p_A - lo_A) * 100, (p_B - lo_B) * 100],
          [(hi_A - p_A) * 100, (hi_B - p_B) * 100]]

ax.errorbar(groups, means, yerr=errors, fmt='o', capsize=10,
            markersize=10, color='darkblue', ecolor='gray', elinewidth=2)
ax.set_ylabel('Конверсия (%)')
ax.set_title('Конверсии с 95% доверительными интервалами')
for i, v in enumerate(means):
    ax.text(i + 0.05, v, f'{v:.2f}%', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

if lo_B > hi_A:
    print('\nДоверительные интервалы НЕ пересекаются — есть основания считать B лучше.')
else:
    print('\nДоверительные интервалы пересекаются — нужна формальная проверка гипотез.')

## 4. Проверка гипотез: z-тест для двух пропорций

Формулируем гипотезы:
- $H_0$: $p_A = p_B$ (новая версия не влияет на конверсию)
- $H_1$: $p_A \ne p_B$ (конверсии различаются)

Статистика z-теста для двух пропорций:

$$z = \frac{\hat{p}_B - \hat{p}_A}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_A}+\frac{1}{n_B}\right)}}, \quad \hat{p} = \frac{x_A + x_B}{n_A + n_B}$$

In [ ]:
# z-тест для двух пропорций
p_pool = (s_A + s_B) / (n_A + n_B)
se_pool = np.sqrt(p_pool * (1 - p_pool) * (1/n_A + 1/n_B))
z_stat = (p_B - p_A) / se_pool
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

alpha = 0.05
print('z-тест для двух пропорций')
print('=' * 60)
print(f'Объединённая пропорция p_pool: {p_pool:.4f}')
print(f'z-статистика: {z_stat:.4f}')
print(f'p-значение: {p_value:.4f}')
print(f'Уровень значимости alpha: {alpha}')
print('-' * 60)
if p_value < alpha:
    print('Отвергаем H0: различие статистически значимо.')
else:
    print('Не отвергаем H0: различие не значимо.')

# Проверим с помощью хи-квадрат теста (эквивалентно для 2x2)
contingency = np.array([[s_A, n_A - s_A], [s_B, n_B - s_B]])
chi2, p_chi2, dof, expected = stats.chi2_contingency(contingency, correction=False)
print(f'\nПроверка хи-квадрат тестом: chi2 = {chi2:.4f}, p = {p_chi2:.4f}')
print(f'(z^2 = {z_stat**2:.4f} должно совпадать с chi2)')

In [ ]:
# Визуализация распределения z-статистики
fig, ax = plt.subplots(figsize=(10, 6))
x = np.linspace(-4, 4, 1000)
ax.plot(x, stats.norm.pdf(x), 'b-', label='N(0,1) при H0')

z_crit = stats.norm.ppf(1 - alpha / 2)
ax.fill_between(x, 0, stats.norm.pdf(x), where=(np.abs(x) >= z_crit),
                color='red', alpha=0.3, label='Область отклонения H0')
ax.axvline(z_stat, color='green', linestyle='--', linewidth=2,
           label=f'z = {z_stat:.2f}')
ax.axvline(-z_crit, color='red', linestyle=':', alpha=0.7)
ax.axvline(z_crit, color='red', linestyle=':', alpha=0.7)
ax.set_xlabel('z')
ax.set_ylabel('Плотность')
ax.set_title('z-тест: наблюдаемая статистика и критическая область')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Размер эффекта и мощность теста

p-значение говорит о статистической значимости, но не о **величине** эффекта. Оценим размер эффекта (Cohen's h) и мощность теста.

In [ ]:
# Размер эффекта Cohen's h для пропорций
def cohens_h(p1, p2):
    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

h = abs(cohens_h(p_B, p_A))
print(f"Размер эффекта Cohen's h: {h:.4f}")
if h < 0.2:
    print('Интерпретация: маленький эффект')
elif h < 0.5:
    print('Интерпретация: средний эффект')
else:
    print('Интерпретация: большой эффект')

# Оценка мощности теста (Монте-Карло)
np.random.seed(0)
n_sim = 2000
reject = 0
for _ in range(n_sim):
    a = np.random.binomial(1, true_conv_A, n_A)
    b = np.random.binomial(1, true_conv_B, n_B)
    xa, xb = a.sum(), b.sum()
    pp = (xa + xb) / (n_A + n_B)
    se = np.sqrt(pp * (1 - pp) * (1/n_A + 1/n_B))
    z = (xb/n_B - xa/n_A) / se
    if 2 * (1 - stats.norm.cdf(abs(z))) < alpha:
        reject += 1
power = reject / n_sim
print(f'\nОценённая мощность теста (Монте-Карло): {power*100:.1f}%')
print('Рекомендуемый порог мощности: 80%')

## 6. Байесовский взгляд на A/B-тест (Beta-Binomial)

Используем сопряжённое априорное распределение **Beta** для конверсии. При наблюдении $x$ успехов из $n$ испытаний апостериорное распределение:

$$p \mid \text{data} \sim \text{Beta}(\alpha_0 + x, \; \beta_0 + n - x)$$

Возьмём неинформативное априорное $\text{Beta}(1, 1)$ и оценим $P(p_B > p_A)$ методом Монте-Карло.

In [ ]:
# Байесовский A/B: Beta-Binomial
alpha0, beta0 = 1, 1  # неинформативное априорное Beta(1,1)

post_A = stats.beta(alpha0 + s_A, beta0 + n_A - s_A)
post_B = stats.beta(alpha0 + s_B, beta0 + n_B - s_B)

# Сэмплирование из апостериорных распределений
np.random.seed(7)
samples_A = post_A.rvs(100000)
samples_B = post_B.rvs(100000)
prob_B_better = (samples_B > samples_A).mean()
expected_uplift = (samples_B - samples_A).mean()

print('Байесовский анализ A/B-теста')
print('=' * 60)
print(f'P(конверсия B > конверсия A) = {prob_B_better*100:.2f}%')
print(f'Ожидаемый прирост конверсии: {expected_uplift*100:.2f} п.п.')

# Визуализация апостериорных распределений
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.linspace(0.09, 0.17, 500)
axes[0].plot(x, post_A.pdf(x), label='A (контроль)', color='steelblue')
axes[0].plot(x, post_B.pdf(x), label='B (новая)', color='coral')
axes[0].fill_between(x, post_A.pdf(x), alpha=0.3, color='steelblue')
axes[0].fill_between(x, post_B.pdf(x), alpha=0.3, color='coral')
axes[0].set_xlabel('Конверсия p')
axes[0].set_ylabel('Плотность апостериорного распределения')
axes[0].set_title('Апостериорные распределения конверсий')
axes[0].legend()

diff = samples_B - samples_A
axes[1].hist(diff, bins=60, color='green', alpha=0.7, edgecolor='black')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='Нет разницы')
axes[1].set_xlabel('Разница конверсий (B - A)')
axes[1].set_ylabel('Частота')
axes[1].set_title(f'Распределение разницы: P(B>A)={prob_B_better*100:.1f}%')
axes[1].legend()
plt.tight_layout()
plt.show()

## 7. Выводы и рекомендации

Соберём результаты классического и байесовского подходов в единый вывод.

In [ ]:
print('ИТОГОВЫЙ ОТЧЁТ ПО A/B-ТЕСТУ')
print('=' * 60)
print(f'Конверсия A: {p_A*100:.2f}%  (n={n_A})')
print(f'Конверсия B: {p_B*100:.2f}%  (n={n_B})')
print(f'Абсолютный прирост: {(p_B-p_A)*100:.2f} п.п.')
print(f'Относительный прирост: {(p_B/p_A-1)*100:.1f}%')
print('-' * 60)
print(f'z-тест p-значение: {p_value:.4f}')
print(f"Размер эффекта (Cohen's h): {h:.3f}")
print(f'P(B > A) по Байесу: {prob_B_better*100:.1f}%')
print('-' * 60)
if p_value < alpha and prob_B_better > 0.95:
    print('РЕКОМЕНДАЦИЯ: Внедрить версию B — эффект статистически значим')
    print('и подтверждён байесовским анализом.')
else:
    print('РЕКОМЕНДАЦИЯ: Недостаточно данных для уверенного решения,')
    print('стоит продолжить сбор данных.')

## Упражнения

### Упражнение 1: Односторонний тест
1. Переформулируйте гипотезу как одностороннюю ($H_1: p_B > p_A$)
2. Как изменится p-значение и вывод?

### Упражнение 2: Анализ вторичной метрики
1. Проверьте гипотезу о равенстве среднего времени на сайте (t-тест)
2. Постройте доверительный интервал для разницы средних

### Упражнение 3: Планирование размера выборки
1. Какой размер выборки нужен для мощности 80% при эффекте +1.5 п.п.?
2. Используйте `statsmodels` или формулу для пропорций

### Упражнение 4: Влияние априорного распределения
1. Повторите байесовский анализ с информативным априорным Beta(12, 88)
2. Как меняется $P(B>A)$ при малом объёме данных (первые 200 пользователей)?

---

**Решения** можно найти в ноутбуке `solutions/16_Solutions.ipynb`